# 1958 — Frank Rosenblatt
## Perceptron: เซลล์ประสาทที่ "เรียนรู้" ได้เป็นครั้งแรก

| | |
|---|---|
| **ผู้คิดค้น** | Frank Rosenblatt — นักจิตวิทยา, Cornell Aeronautical Laboratory |
| **ผลงาน** | *"The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain"* — Psychological Review, 1958 |
| **ฮาร์ดแวร์** | **Mark I Perceptron** — เครื่องจริงที่มีกล้องเซลล์รับแสง 20×20 = 400 จุด และ weight ปรับด้วยตัวต้านทานปรับค่าได้ที่มอเตอร์หมุนเอง |
| **ต่างจาก 1943 อย่างไร** | มี **weight เป็นจำนวนจริง** + **bias** และที่สำคัญที่สุด: **ปรับ weight เองจากตัวอย่าง** (supervised learning) |

## 1. แบบจำลอง Perceptron

$$
z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}\cdot\mathbf{x} + b
\qquad\qquad
\hat{y} = \text{step}(z) = \begin{cases} 1 & z \ge 0 \\ 0 & z < 0 \end{cases}
$$

### กฎการเรียนรู้ (Perceptron Learning Rule)

ทุกครั้งที่เห็นตัวอย่าง $(\mathbf{x}, y)$:

$$
\text{error} = y - \hat{y}, \qquad
\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot \text{error} \cdot \mathbf{x}, \qquad
b \leftarrow b + \eta \cdot \text{error}
$$

| กรณี | error | สิ่งที่เกิดขึ้น |
|---|:-:|---|
| ทายถูก | 0 | ไม่เปลี่ยนอะไร |
| ควรเป็น 1 แต่ทาย 0 | +1 | **บวก** $\mathbf{x}$ เข้า weight → ครั้งหน้า $z$ ใหญ่ขึ้น |
| ควรเป็น 0 แต่ทาย 1 | −1 | **ลบ** $\mathbf{x}$ ออกจาก weight → ครั้งหน้า $z$ เล็กลง |

$\eta$ (eta) คือ **learning rate** — ขนาดก้าวของการปรับ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def train_perceptron(X, y, lr=1.0, epochs=20):
    w, b = np.zeros(X.shape[1]), 0.0
    log, history = [], []
    for epoch in range(1, epochs + 1):
        errors = 0
        for xi, yi in zip(X, y):
            z = w @ xi + b
            y_hat = int(z >= 0)
            err = yi - y_hat
            if err != 0:
                w = w + lr * err * xi
                b = b + lr * err
                errors += 1
            log.append({"epoch": epoch, "x1": xi[0], "x2": xi[1], "y": yi, "z": z,
                        "y_hat": y_hat, "error": err, "w1": w[0], "w2": w[1], "b": b})
        history.append({"epoch": epoch, "w": w.copy(), "b": b, "errors": errors})
        if errors == 0:
            break
    return w, b, history, pd.DataFrame(log)


def predict(X, w, b):
    return (X @ w + b >= 0).astype(int)

## 2. ตัวอย่างการคำนวณด้วยมือ: สอน Perceptron ให้เรียนรู้ AND

เริ่มต้น $\mathbf{w} = (0, 0)$, $b = 0$, $\eta = 1$ ป้อนข้อมูลตามลำดับ (0,0) → (0,1) → (1,0) → (1,1)

### Epoch 1

| ขั้น | $\mathbf{x}$ | $y$ | $z = \mathbf{w}\cdot\mathbf{x} + b$ | $\hat{y}$ | error | อัปเดต → $\mathbf{w}$, $b$ |
|:-:|:-:|:-:|---|:-:|:-:|---|
| 1 | (0,0) | 0 | $0+0+0 = 0$ | 1 | **−1** | $\mathbf{w} = (0,0) - (0,0) = (0,0)$, $b = 0 - 1 = -1$ |
| 2 | (0,1) | 0 | $0+0-1 = -1$ | 0 | 0 | ไม่เปลี่ยน |
| 3 | (1,0) | 0 | $0+0-1 = -1$ | 0 | 0 | ไม่เปลี่ยน |
| 4 | (1,1) | 1 | $0+0-1 = -1$ | 0 | **+1** | $\mathbf{w} = (0,0) + (1,1) = (1,1)$, $b = -1 + 1 = 0$ |

### Epoch 2

| ขั้น | $\mathbf{x}$ | $y$ | $z$ | $\hat{y}$ | error | อัปเดต → $\mathbf{w}$, $b$ |
|:-:|:-:|:-:|---|:-:|:-:|---|
| 1 | (0,0) | 0 | $0+0+0 = 0$ | 1 | **−1** | $\mathbf{w} = (1,1)$, $b = -1$ |
| 2 | (0,1) | 0 | $0+1-1 = 0$ | 1 | **−1** | $\mathbf{w} = (1,1) - (0,1) = (1,0)$, $b = -2$ |
| 3 | (1,0) | 0 | $1+0-2 = -1$ | 0 | 0 | ไม่เปลี่ยน |
| 4 | (1,1) | 1 | $1+0-2 = -1$ | 0 | **+1** | $\mathbf{w} = (1,0) + (1,1) = (2,1)$, $b = -1$ |

ทำแบบนี้ต่อไปเรื่อยๆ จน **ทั้ง epoch ไม่มี error เลย** = เรียนรู้สำเร็จ
ลองทำ Epoch 3 ด้วยตัวเองก่อน แล้วรันโค้ดด้านล่างเพื่อเช็คคำตอบ 👇

In [ ]:
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

w, b, hist_and, log_and = train_perceptron(X_and, y_and, lr=1.0)
print(log_and.to_string(index=False, float_format=lambda v: f"{v:g}"))
print(f"\nเรียนรู้สำเร็จใน epoch ที่ {hist_and[-1]['epoch']}:  w = {w},  b = {b:g}")
print("ตรวจคำตอบ:", predict(X_and, w, b), " (เป้าหมาย", y_and, ")")

### ตรวจคำตอบสุดท้ายด้วยมือ

ได้ $\mathbf{w} = (2, 1)$, $b = -3$ → เส้นแบ่งคือ $2x_1 + x_2 - 3 = 0$

| $\mathbf{x}$ | $z = 2x_1 + x_2 - 3$ | $\hat{y}$ | ถูก? |
|:-:|---|:-:|:-:|
| (0,0) | $0 + 0 - 3 = -3$ | 0 | ✅ |
| (0,1) | $0 + 1 - 3 = -2$ | 0 | ✅ |
| (1,0) | $2 + 0 - 3 = -1$ | 0 | ✅ |
| (1,1) | $2 + 1 - 3 = 0$ | 1 | ✅ |

> สังเกต: คำตอบไม่จำเป็นต้องเป็น (1,1,−1.5) แบบที่คนออกแบบเอง — Perceptron หาเส้นแบ่ง **เส้นไหนก็ได้** ที่แยกข้อมูลได้ถูก

### ดูเส้นแบ่งเคลื่อนที่ในแต่ละ epoch

In [ ]:
def plot_line(ax, w, b, **kw):
    xs = np.linspace(-0.5, 1.5, 50)
    if abs(w[1]) > 1e-12:
        ax.plot(xs, -(w[0] * xs + b) / w[1], **kw)
    else:
        ax.axvline(-b / w[0], **kw)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0, 1, len(hist_and)))
for h, col in zip(hist_and, colors):
    plot_line(ax, h["w"], h["b"], color=col, lw=2,
              label=f"epoch {h['epoch']}: w=({h['w'][0]:g},{h['w'][1]:g}), b={h['b']:g}")
for (x1, x2), t in zip(X_and, y_and):
    ax.scatter(x1, x2, s=220, c="tab:green" if t else "tab:red", edgecolors="k", zorder=3)
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x1", ylabel="x2", title="AND: decision boundary after each epoch")
ax.set_aspect("equal")
ax.legend(fontsize=7, loc="upper right")

axes[1].bar([h["epoch"] for h in hist_and], [h["errors"] for h in hist_and], color="tab:blue")
axes[1].set(xlabel="epoch", ylabel="mistakes in epoch", title="Mistakes per epoch (0 = learned)")
plt.tight_layout()
plt.show()

## 3. ตัวอย่างที่ใหญ่ขึ้น: แยกข้อมูล 2 กลุ่ม

**Perceptron Convergence Theorem** (Novikoff, 1962): ถ้าข้อมูล **แยกได้ด้วยเส้นตรง (linearly separable)** กฎการเรียนรู้จะหาเส้นแบ่งเจอเสมอในจำนวนรอบที่จำกัด

ทดลองกับข้อมูลสุ่ม 100 จุด แบ่งเป็น 2 กลุ่ม:

In [ ]:
rng = np.random.default_rng(0)
X0 = rng.normal([-2, -1], 0.7, size=(50, 2))
X1 = rng.normal([2, 1], 0.7, size=(50, 2))
X_blob = np.vstack([X0, X1])
y_blob = np.array([0] * 50 + [1] * 50)
order = rng.permutation(100)
X_blob, y_blob = X_blob[order], y_blob[order]

w_b, b_b, hist_b, _ = train_perceptron(X_blob, y_blob, lr=0.1, epochs=100)
acc = (predict(X_blob, w_b, b_b) == y_blob).mean()
print(f"จบใน {len(hist_b)} epoch   w = {np.round(w_b, 3)}   b = {b_b:.3f}   accuracy = {acc:.0%}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(*X_blob[y_blob == 0].T, c="tab:red", label="class 0", edgecolors="k")
ax.scatter(*X_blob[y_blob == 1].T, c="tab:green", label="class 1", edgecolors="k")
xs = np.linspace(-4.5, 4.5, 50)
ax.plot(xs, -(w_b[0] * xs + b_b) / w_b[1], "k--", lw=2, label="learned boundary")
ax.set(xlim=(-4.5, 4.5), ylim=(-4, 4), xlabel="x1", ylabel="x2", title="Perceptron on linearly separable data")
ax.legend()
plt.show()

## 4. ขีดจำกัด: Perceptron เรียน XOR ไม่ได้

ในปี **1969** Marvin Minsky และ Seymour Papert ตีพิมพ์หนังสือ *"Perceptrons"* พิสูจน์ว่า Perceptron ชั้นเดียว **แก้ปัญหาที่แยกด้วยเส้นตรงไม่ได้** เช่น XOR
ความผิดหวังนี้เป็นหนึ่งในสาเหตุของ **"AI Winter" ครั้งแรก** — งบวิจัย Neural Network ลดลงไปเกือบทศวรรษ

ลองสอน XOR ดู — error จะไม่มีวันเป็น 0:

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

_, _, hist_xor, _ = train_perceptron(X_xor, y_xor, lr=1.0, epochs=30)
errs = [h["errors"] for h in hist_xor]
print("จำนวนครั้งที่ผิดในแต่ละ epoch:", errs)
print("มี epoch ไหนผิด 0 ครั้งไหม?", 0 in errs)

plt.figure(figsize=(7, 3.5))
plt.bar(range(1, len(errs) + 1), errs, color="tab:red")
plt.xlabel("epoch"); plt.ylabel("mistakes")
plt.title("XOR: single perceptron never converges")
plt.show()

## 5. Multi-Layer Neural Network

ทางออกคือ **เพิ่มชั้นซ่อน (hidden layer)** — Rosenblatt เองก็เขียนถึง perceptron หลายชั้นไว้ในหนังสือ *Principles of Neurodynamics* (1962)
แต่สมัยนั้น **ยังไม่มีวิธีที่ดีในการเรียนรู้ weight ของชั้นซ่อน** เพราะกฎ Perceptron ต้องรู้ว่า "คำตอบที่ถูก" ของแต่ละเซลล์คืออะไร ซึ่งชั้นซ่อนไม่มี
ปัญหานี้แก้ได้ในปี **1986** ด้วย **Backpropagation** (Rumelhart, Hinton & Williams)

### คำนวณด้วยมือ: เครือข่าย 2 ชั้นที่แก้ XOR (กำหนด weight เอง)

$$
h_1 = \text{step}(x_1 + x_2 - 0.5) \;\;(\text{OR}) \qquad
h_2 = \text{step}(x_1 + x_2 - 1.5) \;\;(\text{AND}) \qquad
y = \text{step}(h_1 - h_2 - 0.5)
$$

| $\mathbf{x}$ | $z_{h1} = x_1+x_2-0.5$ | $h_1$ | $z_{h2} = x_1+x_2-1.5$ | $h_2$ | $z_y = h_1-h_2-0.5$ | $y$ |
|:-:|---|:-:|---|:-:|---|:-:|
| (0,0) | $-0.5$ | 0 | $-1.5$ | 0 | $0-0-0.5 = -0.5$ | **0** |
| (0,1) | $0.5$ | 1 | $-0.5$ | 0 | $1-0-0.5 = 0.5$ | **1** |
| (1,0) | $0.5$ | 1 | $-0.5$ | 0 | $1-0-0.5 = 0.5$ | **1** |
| (1,1) | $1.5$ | 1 | $0.5$ | 1 | $1-1-0.5 = -0.5$ | **0** |

**เคล็ดลับ:** ชั้นซ่อน **ย้ายจุดไปอยู่ในพื้นที่ใหม่** $(h_1, h_2)$ ที่แยกด้วยเส้นตรงได้แล้ว — (0,1) และ (1,0) ถูกรวมไปเป็นจุดเดียวกันคือ (1,0)

In [ ]:
W1 = np.array([[1, 1], [1, 1]])
b1 = np.array([-0.5, -1.5])
W2 = np.array([1, -1])
b2 = -0.5

step = lambda z: (z >= 0).astype(int)
H = step(X_xor @ W1.T + b1)
Y = step(H @ W2 + b2)

print(pd.DataFrame({"x1": X_xor[:, 0], "x2": X_xor[:, 1], "h1": H[:, 0], "h2": H[:, 1],
                    "y": Y, "target": y_xor}).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
xs = np.linspace(-0.5, 1.5, 50)
ax = axes[0]
ax.plot(xs, 0.5 - xs, "b--", label="h1: x1 + x2 = 0.5")
ax.plot(xs, 1.5 - xs, "m--", label="h2: x1 + x2 = 1.5")
for (a, b_), t in zip(X_xor, y_xor):
    ax.scatter(a, b_, s=220, c="tab:green" if t else "tab:red", edgecolors="k", zorder=3)
    ax.annotate(f"({a},{b_})", (a, b_), textcoords="offset points", xytext=(10, 8))
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x1", ylabel="x2", title="Input space: XOR needs 2 lines")
ax.set_aspect("equal"); ax.legend(fontsize=8, loc="lower left")

ax = axes[1]
ax.plot(xs, xs - 0.5, "k--", label="output: h1 - h2 = 0.5")
for (a, b_), (h1, h2), t in zip(X_xor, H, y_xor):
    ax.scatter(h1, h2, s=220, c="tab:green" if t else "tab:red", edgecolors="k", zorder=3)
    ax.annotate(f"x=({a},{b_})", (h1, h2), textcoords="offset points", xytext=(10, -14 if (a, b_) == (1, 0) else 8))
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="h1", ylabel="h2", title="Hidden space: now 1 line is enough")
ax.set_aspect("equal"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

### (ตัวอย่างล่วงหน้า) ให้เครื่องหา weight เองด้วย Backpropagation — วิธีของปี 1986

เปลี่ยน step เป็น **sigmoid** $\sigma(z) = \frac{1}{1+e^{-z}}$ ซึ่งหาอนุพันธ์ได้ แล้วใช้ **chain rule** ส่ง error ย้อนจากเอาต์พุตไปยังชั้นซ่อน

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
rng = np.random.default_rng(1)
Wa, ba = rng.normal(0, 1, (2, 4)), np.zeros(4)
Wb, bb = rng.normal(0, 1, (4, 1)), np.zeros(1)
Xf, yf = X_xor.astype(float), y_xor.reshape(-1, 1).astype(float)

losses = []
for it in range(5000):
    h = sigmoid(Xf @ Wa + ba)
    out = sigmoid(h @ Wb + bb)
    losses.append(float(np.mean(-(yf * np.log(out) + (1 - yf) * np.log(1 - out)))))
    d_out = (out - yf) / len(Xf)
    d_h = (d_out @ Wb.T) * h * (1 - h)
    Wb -= 1.0 * h.T @ d_out;  bb -= 1.0 * d_out.sum(0)
    Wa -= 1.0 * Xf.T @ d_h;   ba -= 1.0 * d_h.sum(0)

print("ความน่าจะเป็นที่ทำนาย:", out.ravel().round(3))
print("ปัดเป็นคำตอบ:          ", (out.ravel() >= 0.5).astype(int), " เป้าหมาย:", y_xor)

plt.figure(figsize=(7, 3.5))
plt.plot(losses)
plt.xlabel("iteration"); plt.ylabel("cross-entropy loss")
plt.title("2-layer network learns XOR with backpropagation")
plt.show()

## 6. แบบฝึกหัด

1. ทำ **Epoch 3** ของตัวอย่าง AND ด้วยมือ แล้วเทียบกับตาราง log ด้านบน
2. สอน Perceptron ให้เรียน **OR** (`y = [0, 1, 1, 1]`) — ใช้กี่ epoch? ได้ weight เท่าไร? ตรวจด้วยมือว่าถูกทุกจุด
3. ลองเปลี่ยน learning rate เป็น `0.1` ในตัวอย่าง AND — จำนวน epoch และ "รูปร่าง" ของเส้นแบ่งเปลี่ยนไหม? (ใบ้: เมื่อเริ่มจาก 0 การคูณ $\eta$ ทั้งหมดแค่ย่อ/ขยายขนาดของ $\mathbf{w}, b$)
4. เปลี่ยนตัวอย่างข้อมูล 2 กลุ่มให้ศูนย์กลางใกล้กันจนทับกัน — Perceptron ยังจบได้ไหม?

## 7. สรุป

| ปี | เหตุการณ์ |
|---|---|
| 1958 | Rosenblatt เสนอ Perceptron + กฎการเรียนรู้ |
| 1962 | Novikoff พิสูจน์ว่าเรียนรู้สำเร็จเสมอถ้าข้อมูลแยกด้วยเส้นตรงได้ |
| 1969 | Minsky & Papert ชี้ขีดจำกัด (XOR) → AI Winter |
| 1986 | Backpropagation ทำให้ฝึก Multi-Layer Network ได้จริง |

**ต่อไป:** ระหว่างที่ Neural Network ซบเซา นักฟิสิกส์ John Hopfield เสนอมุมมองใหม่ในปี 1982 — เครือข่ายที่ **"จำ" รูปแบบได้** เหมือนหน่วยความจำ (ดู `1982.ipynb`)